# 将网络集导出为通用 MDIF 文件

通常，在改变一些其他参数（如温度、电压、电流等）时，会记录一组`Networks`。一旦获取了这组数据，有时将所有网络组合成一个单独的通用 MDIF 文件，以便在 AWR Microwave Office 等 CAD 工具中使用，会很有用。

In [ ]:
import pathlib
import tempfile
import zipfile

import numpy as np
import requests

import skrf as rf

## Narda 3752 相移器

在本示例中，我们正在对一款旧的 [narda 相移器 3752](https://nardamiteq.com/docs/119-PHASESHIFTERS.PDF) 在 1.5 GHz 频率下进行表征。![narda 3752 相移器](phase_shifter_measurements/Narda_3752.jpg)为了确定在该特定频率下可以获得的相移值，我们测量了 1-2 GHz 频段内相移旋钮的 19 个位置（从 0 到 180）的散射参数。这些测量结果被加载到 [NetworkSets](../../tutorials/NetworkSet.ipynb) 对象中：

In [ ]:
# Array containing the 19 phase shift indicator values
indicators_mes = np.linspace(0, 180, num=19)  # from 0 to 180 per 10

In [ ]:
ntw_set = rf.networkSet.NetworkSet.from_zip('phase_shifter_measurements/phase_shifter_measurements.zip')
print('ntw_set contains', len(ntw_set), 'networks')

NetworkSet 的内容可以导出为 MDIF 文件：

In [ ]:
ntw_set.write_mdif("phase_shifter.mdif")

请注意，可以通过传递可选参数 `values` 和 `data_types` 来调整 MDIF 文件中定义的参数、值和类型。因此，可以将“indicator”定义为类型为“double”的 MDIF 变量，并将 NetworkSet 保存为“phase_shifter.mdif”：

In [ ]:
values = {"indicator": indicators_mes}
data_types = {"indicator": "double"}
ntw_set.write_mdif("phase_shifter.mdif", values, data_types)

## ADRF5720[ADRF5720](https://www.analog.com/en/products/adrf5720.html) 是一款 6 位 9 kHz 至 40 GHz 数字步进衰减器，并且在不同温度下进行了测量，从而生成了许多 Touchstone 文件。

In [ ]:
# download the zip archive of ADRF5720 Touchstone files
url = "https://www.analog.com/media/en/simulation-models/s-parameters/ADRF5720_Sparameters.zip"
try:
    response = requests.get(url, timeout=10)
    open("ADRF5720.zip", "wb").write(response.content)
    tmpdir = pathlib.Path(tempfile.mkdtemp())
    zf = zipfile.ZipFile("ADRF5720.zip")
    zf.extractall(path = tmpdir)
    zf.close()

    # filter out one file (which contains '5720_noDC')
    input_files = [file for file in tmpdir.rglob('*.s2p') if '5720_noDC' not in file.stem]
    ns = rf.NetworkSet(rf.read_all(files=[str(file) for file in input_files]))
    print('ns contains', len(ns), 'networks')

    # extract the attenuation value from the filenames and store in list
    attn = []
    temp = []
    for f in input_files:
        _,_,_,_,_,a,t = f.stem.split('_')
        t = t.replace('M','-').replace('C','')
        attn.append(float(a))
        temp.append(int(t))

    # sort files
    v = list(zip(attn,temp,input_files))
    v.sort()
    (attn,temp,input_files) = list(zip(*v))

    values = {'ATTEN': attn, 'TEMP_C': temp }
    datatypes = {'ATTEN': 'double', 'TEMP_C': 'double'}

    # write to a generalized MDIF file
    ns.write_mdif("ADRF5720.mdif", values, datatypes)

except requests.ReadTimeout:
    print('Timeout... skipping this example')

最后，可以将参数化的 MDIF 文件导入到 AWR Microwave Office 中：![AWR](import_mdif_to_awr_mwo.png)